In [1]:
import os
from pathlib import Path

os.chdir(Path.cwd().parent)

print(Path.cwd())

d:\private\ai-research-paper-assistant


In [2]:
import json
from tqdm import tqdm
import pandas as pd
from src.database.pgvector_storage import PGVectorStore
from src.embedder.ollama_embedder import OllamaEmbedder
from src.evaluation.metrics import compute_mean_gen_results
from src.evaluation.evaluator import Evaluator
from src.utils.helpers import build_scifact_data
from langchain_core.documents import Document

c:\Users\luann\miniconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
db = PGVectorStore()
db.delete()
print(db.count())

0


In [8]:
corpus, queries, qrels = build_scifact_data()
documents = []
for k, v in tqdm(corpus.items(), desc="READING"):
    doc = Document(
        page_content=v["content"],
        metadata={
            "document_id": k,
            "title": v["title"]
        }
    )
    documents.append(doc)

READING: 100%|██████████| 5183/5183 [00:00<00:00, 340994.44it/s]


In [9]:
emebdder = OllamaEmbedder(dimensions=2046, num_ctx=8192, num_gpu=1)
all_embeddings = emebdder.embed_documents(documents)

In [10]:
print(len(all_embeddings))
print(db.count())

5183
0


In [11]:
db.insert(documents, all_embeddings)

In [12]:
from src.evaluation.evaluator import Evaluator
from src.reranker.cross_encoder_reranker import CrossEncoderReRanker

cross_reranker = CrossEncoderReRanker(device="cuda")

evaluator_dense = Evaluator(repository=db, embedder=emebdder, search_mode="dense")
evaluator_hybrid = Evaluator(repository=db, embedder=emebdder, search_mode="hybrid")
evaluator_hybrid_rerank = Evaluator(repository=db, embedder=emebdder, reranker=cross_reranker, search_mode="hybrid")

Load evaluator OK
Init Successfully!
Load evaluator OK
Init Successfully!
Load evaluator OK
Init Successfully!


In [4]:
db = PGVectorStore()
db.count()

5183

In [5]:
all_chunks = db.get_all_chunks()
all_chunks[0]

{'id': 11222,
 'document_id': '4983',
 'content': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar (1.2 versus 1.1 microm2/ms). Relative anisotropy was 

In [13]:
eval_dataset_path = r"D:\private\ai-research-paper-assistant\data\eval_data.json"
with open(eval_dataset_path, "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)

results_dense = await evaluator_dense.evaluate(queries, qrels, eval_dataset)
results_hybrid = await evaluator_hybrid.evaluate(queries, qrels, eval_dataset)
results_hybrid_rerank = await evaluator_hybrid_rerank.evaluate(queries, qrels, eval_dataset)

PreparingIDS: 100%|██████████| 300/300 [04:21<00:00,  1.15it/s]


Ok


Evaluating:   0%|          | 0/100 [26:14<?, ?it/s]


IncompleteOutputException: 

In [24]:
cnt = 0
for doc in eval_dataset:
    if doc["response"] == "" or not doc["response"]:
        cnt += 1

print(cnt)

93


In [7]:
from src.evaluation.metrics import create_evaluator

llm, emebdder = create_evaluator(embed_model_name="bge-m3")

a = emebdder.embed_query("What is arcface")

print(len(a))


1024


In [7]:
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = await client.chat.completions.create(
    model="llama3.1",
    messages=[
        {
            "role": "user",
            "content": "Return a JSON object containing a short summary."
        }
    ],
    max_tokens=4096
)

print(response.choices[0].finish_reason)
print(response.usage)
print(response.choices[0].message.content)

stop
CompletionUsage(completion_tokens=210, prompt_tokens=19, total_tokens=229, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0))
Here is a simple example of a JSON object in Python:

```python
import json

# Create a dictionary to store the data
data = {
    "summary": "This is a brief description of the project.",
    "title": "My Project",
    "author": "John Doe"
}

# Convert the dictionary to a JSON formatted string
json_data = json.dumps(data)

# Print the JSON object
print(json_data)
```

When you run this code, it will output the following JSON:

```json
{"summary": "This is a brief description of the project.", "title": "My Project", "author": "John Doe"}
```

You can easily use this JSON for further processing or return it from a function as you initially mentioned:

```python
def get_summary():
    data = {
        "summary": "This is a brief description of the project.",
        "title": "My Project",
        "aut